# EXP-003 판정 근거

목적: EXP-003(질의 조건 목록을 RRF 세 번째 목록으로, `min_fields: 2`)의 판정 수치를 SPEC "메타데이터 가설의 판정"(D-18) 순서로 확인하고, multi-doc 결과를 dense·EXP-001·EXP-002와 나란히 본다. 질의 원문은 출력하지 않는다(D-10).

In [1]:
import json
from pathlib import Path

import pandas as pd

from rag.conditions import ConditionIndex
from rag.eval import metrics as single
from rag.eval_multi import metrics as multi
from rag.index import load_cfg as load_hydra

In [2]:
ROOT = Path.cwd().parent
RUNS = ROOT / "data/runs"
SINGLE = {
    "exp001": RUNS / "kure-v1-fixed-512-64-hybrid-dev-full.csv",
    "exp003": RUNS / "kure-v1-fixed-512-64-hybrid-cond-dev-full.csv",
}
MULTI = {
    "dense": RUNS / "kure-v1-fixed-512-64-multidoc.csv",
    "exp001": RUNS / "kure-v1-fixed-512-64-hybrid-multidoc.csv",
    "exp002": RUNS / "kure-v1-fixed-512-64-hybrid-meta-multidoc.csv",
    "exp003": RUNS / "kure-v1-fixed-512-64-hybrid-cond-multidoc.csv",
}
COMPARE = {
    "single": RUNS / "exp003-single-compare.json",
    "multi": RUNS / "exp003-multi-compare.json",
}

## 1단계 single-doc

`rag.eval.compare` 출력(회의 단위 bootstrap 10,000회, seed 20260929). 채택은 95% 구간 하한 > 0이고 점 추정 ≥ +1.0%p, 기각은 상한 < 0이다.

In [3]:
s = json.loads(COMPARE["single"].read_text(encoding="utf-8"))
{k: s[k] for k in ("diff", "ci", "verdict")}

{'diff': 0.0, 'ci': [0.0, 0.0], 'verdict': '보류'}

1단계가 보류라 2단계 multi-doc으로 정한다.

## 2단계 multi-doc

`rag.eval_multi.compare` 출력(`pool_id` 단위 bootstrap 10,000회, seed 20260929, 동결 세트 600건). 채택은 Recall@5 구간 하한 > 0, 점 추정 ≥ +1.0%p, Complete@5 구간 상한 ≥ 0이다.

In [4]:
m = json.loads(COMPARE["multi"].read_text(encoding="utf-8"))
rows = [r for v in m.values() if isinstance(v, list) for r in v]
diffs = pd.DataFrame(rows)
diffs["diff %p"] = (diffs["diff"] * 100).round(2)
diffs["95% 구간 %p"] = diffs["ci"].map(lambda c: [round(x * 100, 2) for x in c])
keep = diffs["metric"].isin(["recall@5", "complete@5", "hit@5", "recall@10"])
diffs.loc[keep, ["group", "value", "n", "metric", "diff %p", "95% 구간 %p"]]

,group,value,n,metric,diff %p,95% 구간 %p
0,all,all,600,recall@5,14.71,"[12.82, 16.64]"
1,all,all,600,complete@5,13.50,"[10.83, 16.33]"
2,all,all,600,hit@5,16.00,"[13.17, 19.0]"
3,all,all,600,recall@10,16.15,"[14.12, 18.21]"
6,type,conf,233,recall@5,11.47,"[8.74, 14.32]"
7,type,conf,233,complete@5,19.74,"[14.59, 24.89]"
8,type,conf,233,hit@5,2.58,"[0.86, 4.72]"
9,type,conf,233,recall@10,9.62,"[6.97, 12.45]"
12,type,law,152,recall@5,22.52,"[18.32, 26.9]"
13,type,law,152,complete@5,11.84,"[6.58, 17.11]"


## multi-doc 순위 네 개 비교

dense(D-17 회복 기준), EXP-001(기준), EXP-002(기각, BM25 메타데이터 접두), EXP-003의 유형별 Recall@5·Complete@5.

In [5]:
hcfg = load_hydra([])
ev = hcfg.multidoc.eval
gold_m = multi.load_gold(ROOT / hcfg.paths.multidoc_frozen)
scores_m = {
    name: multi.per_query(multi.ranked_docs(multi.read_run(p), gold_m, max(ev.ks)), gold_m, ev.ks)
    for name, p in MULTI.items()
}
cols = ["recall@5", "complete@5"]
pd.concat({name: s[cols].join(gold_m["type"]).groupby("type").mean()
           .assign(**{"전체 recall@5": s["recall@5"].mean()})
           for name, s in scores_m.items()}, axis=1).round(4)

dense                          exp001                         \
           recall@5 complete@5 전체 recall@5 recall@5 complete@5 전체 recall@5   
type                                                                         
conf         0.8288     0.6567      0.4657   0.8059     0.6223      0.4336   
law          0.2990     0.1053      0.4657   0.2231     0.0855      0.4336   
questioner   0.1900     0.0930      0.4657   0.1790     0.0930      0.4336   

             exp002                          exp003                         
           recall@5 complete@5 전체 recall@5 recall@5 complete@5 전체 recall@5  
type                                                                        
conf         0.9017     0.7983      0.5868   0.9207     0.8197      0.5808  
law          0.5007     0.2500      0.5868   0.4484     0.2039      0.5808  
questioner   0.3064     0.1628      0.5868   0.3060     0.1721      0.5808

## single-doc에서 조건 목록이 붙은 질의

조건 필드가 2개 이상 뽑힌 dev 질의마다 필드가 2개 이상 맞은 문서가 있었는지(조건 목록이 생겼는지)와 정답 순위를 본다.

In [6]:
with (ROOT / "data/conditions/summary_q.jsonl").open(encoding="utf-8") as f:
    conds = {r["qid"]: r["conditions"] for r in map(json.loads, f)}
gold_s = single.load_gold(single.load_cfg(), "dev-full")
two = [q for q in gold_s.index if len(conds[q]) >= 2]
print("dev-full 질의", len(gold_s), "조건 필드 2개 이상", len(two))

dev-full 질의 3016 조건 필드 2개 이상 7


In [7]:
hcfg3 = load_hydra(["+exp=exp003", "index.scope=full"])
with (ROOT / hcfg3.paths.corpus).open(encoding="utf-8") as f:
    docs = list(map(json.loads, f))
idx = ConditionIndex(docs, hcfg3.retriever.conditions.year_ref_offsets)
pos = {d["doc_id"]: i for i, d in enumerate(docs)}
runs_s = {name: single.read_run(p) for name, p in SINGLE.items()}
ranks_s = {name: single.gold_ranks(r, gold_s, 10) for name, r in runs_s.items()}
order = {name: r.sort_values(["qid", "rank"]).groupby("qid")["doc_id"].apply(tuple)
         for name, r in runs_s.items()}
rows = []
for q in two:
    counts = idx.match_counts(conds[q])
    rows.append({"qid": q, "조건 필드": ", ".join(sorted(conds[q])),
                 "2개 이상 맞은 문서 수": sum(c >= 2 for c in counts.values()),
                 "정답이 맞은 필드 수": counts.get(pos[gold_s.loc[q, "doc_id"]], 0),
                 "순위 바뀜": order["exp001"][q] != order["exp003"][q],
                 "exp001 정답 순위": ranks_s["exp001"][q],
                 "exp003 정답 순위": ranks_s["exp003"][q]})
pd.DataFrame(rows).set_index("qid")

,조건 필드,2개 이상 맞은 문서 수,정답이 맞은 필드 수,순위 바뀜,exp001 정답 순위,exp003 정답 순위
qid,,,,,,
023227-0001,"committee_name, meeting_name",21,2,True,1.0,1.0
029355-0001,"committee_name, names",2,0,False,1.0,1.0
046576-0006,"meeting_name, names",6,1,False,1.0,1.0
049460-0002,"meeting_name, names",4,1,False,1.0,1.0
049776-0018,"date, names",0,0,False,1.0,1.0
050309-0013,"date, names",0,0,False,1.0,1.0
051550-0016,"date, names",0,0,False,1.0,1.0
